# Algoritmos de optimización - Proyecto<br>
Nombre y Apellidos: Manuel Salcedo Alonso y Alejandro López López de la Cova <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---2019/tree/master/SEMINARIO<br>
Problema elegido:
> 1. Sesiones de doblaje <br>


Descripción del problema: Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible.

....

(*) La respuesta es obligatoria





                                        

## Código inicial para instalar librerías y cargar datos

In [25]:
## se importan librerias
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from math import comb, factorial
import itertools, time

# se definen las constantes del problema
NUM_ACTORES = 10
NUM_TOMAS = 30
MAX_TOMAS_DIA = 6


# carga de datos del problema
archivo_datos = "datos_problema_doblaje30_tomas_10_actores.xlsx"
df_datos = pd.read_excel(archivo_datos, header=1)
# ajustes para evitar problemas con la fila "Total"
df_datos["Toma"] = pd.to_numeric(df_datos["Toma"], errors="coerce")
df_datos = df_datos.dropna(subset=["Toma"])
# creacion de columnas de actores
cols_actores = [c for c in df_datos.columns if isinstance(c, (int, float)) and not pd.isna(c)]

# array de actores con booleanos en vez de 0s y 1s
tomas = df_datos[cols_actores].to_numpy(dtype=bool)

# se verifica el tamaño del array (dimensiones correctas)
assert tomas.shape == (NUM_TOMAS, NUM_ACTORES), "Las dimensiones de la matriz de datos no son válidas"
print(f"Los datos se han cargado correctamente. Hay {NUM_TOMAS} tomas, {NUM_ACTORES} actores, y {MAX_TOMAS_DIA} tomas/día como máximo")

Los datos se han cargado correctamente. Hay 30 tomas, 10 actores, y 6 tomas/día como máximo


(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones?




Respuesta

## Sin tener en cuenta las restricciones

Las posibilidades dependen de la forma elegida para representar una solución. En nuestro caso, hemos optado por representar cada solución como una permutación de las 30 tomas, troceadas en bloques consecutivos de 6 donde cada bloque corresponde a un día. El enunciado no especifica el número de días de grabación disponibles, únicamente indica que ningún día puede superar las 6 tomas, por lo que para poder contar el espacio de soluciones de forma cerrada hemos fijado el número de días en 5, que es el mínimo posible dada esa restricción (30/6 = 5).

Esto obliga a que todos los días queden completamente llenos, aunque existe la posibilidad de que una solución con más de 5 días permita agrupar mejor a los actores que comparten toma y obtenga un coste menor que la mejor solución posible con exactamente 5 días. El modelo general, en el que el número de días sería una variable de decisión, es bastante más difícil de contar de forma cerrada, ya que equivale a repartir un conjunto en bloques de tamaño como mucho 6 sin que exista una fórmula simple para ello. Dejamos esa generalización apuntada como posible línea de ampliación para más adelante.

Cabe señalar que el valor D=5 se fija igual en ambos conteos (con y sin restricciones), como parámetro del escenario y no como parte de la restricción que diferencia ambas preguntas. La única restricción que distingue ambos conteos es el máximo de 6 tomas por día.

Con 5 días fijos, calculamos el número de posibilidades sin tener en cuenta la restricción de como máximo 6 tomas por día simplemente contando de cuántas formas se puede asignar cada una de las 30 tomas a cualquiera de los 5 días, de forma independiente para cada toma. Al ser 5 opciones para cada una de las 30 tomas, el resultado es 5 elevado a 30:

> 5^30 = 931.322.574.615.478.515.625

## Teniendo en cuenta las restricciones

Para calcular el número de posibilidades teniendo en cuenta el tope de 6 tomas por día, aprovechamos que, al ser 5 días y haber exactamente 30 tomas, la única forma de repartirlas sin superar el tope en ninguno es que cada día reciba exactamente 6 tomas. Lo abordamos como un proceso de elección paso a paso: primero decidimos qué 6 tomas de las 30 disponibles le tocan al primer día, después, de las 24 que quedan, elegimos las 6 que le tocan al segundo día, y así sucesivamente hasta que no quede ninguna por asignar. Este tipo de conteo es el que se conoce en combinatoria como coeficiente multinomial, y es con el que obtenemos el siguiente valor:

> 30! / (6!)^5 = 1.370.874.167.589.326.400

Conviene indicar que este conteo trata los cinco días como "etiquetados", es decir, considera distinto asignar un grupo de tomas al día 1 y otro al día 2 que a la inversa. Sin embargo, para el coste del problema el día concreto es irrelevante, ya que lo que se paga es el número de días distintos a los que se desplaza cada actor, no en qué jornada ocurre. Por lo tanto, cada agrupación realmente distinta aparece contada 5! = 120 veces (una por cada forma de ordenar los cinco días), por lo que el número de soluciones diferentes sería:

> 30! / ((6!)^5 · 5!) = 11.423.951.396.577.720

Nuestra representación introduce una redundancia: el orden de las tomas dentro de un mismo bloque tampoco afecta al coste, por lo que cada agrupación con días etiquetados está representada (6!)^5 veces adicionales por las permutaciones internas de cada bloque.

Mantenemos el conteo con días etiquetados como valor de referencia porque se corresponde con el espacio que recorre nuestra representación (los bloques de la permutación ocupan posiciones fijas, equivalentes a etiquetas de día), pero dejamos constancia de la simetría anterior porque reduce de forma notable el número de soluciones genuinamente distintas.

In [26]:
'''
Cálculo del número de posibilidades (combinatoria del espacio de soluciones),
con y sin la restricción de máximo 6 tomas por día
'''
D = 5  # fijamos a 5 el numero de dias

# diferenciamos por caso
sin_restricciones = D ** NUM_TOMAS
con_restricciones = factorial(NUM_TOMAS) // (factorial(MAX_TOMAS_DIA) ** D)

# comprobamos el calculo de elegir 6 de los que quedan, dia a dia
restantes = NUM_TOMAS
acumulador = 1
for _ in range(D):
    acumulador *= comb(restantes, MAX_TOMAS_DIA)
    restantes -= MAX_TOMAS_DIA
assert acumulador == con_restricciones

# soluciones distintas, descontando el orden de los 5 dias (÷5!)
distintas = con_restricciones // factorial(D)

print(f"Sin restricciones (5^30): {sin_restricciones:,}")
print(f"Con restricciones (30!/(6!)^5): {con_restricciones:,}")
print(f"Distintas (÷5!): {distintas:,}")
print(f"Factor de reducción (de sin a con restricciones): {sin_restricciones/con_restricciones:,.1f}x")

Sin restricciones (5^30): 931,322,574,615,478,515,625
Con restricciones (30!/(6!)^5): 1,370,874,167,589,326,400
Distintas (÷5!): 11,423,951,396,577,720
Factor de reducción (de sin a con restricciones): 679.4x


Modelo para el espacio de soluciones
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)

Respuesta

Inicialmente valoramos una representación alternativa consistente en un vector de 30 posiciones donde cada posición indicaba directamente el día asignado a esa toma (por ejemplo, [1,1,3,2,...]). Esta opción era más directa de leer, pero tenía un problema para el algoritmo que queremos usar: un cambio puntual (modificar el día de una sola toma) puede dejar un día con más de 6 tomas, rompiendo la restricción del enunciado. Habría que comprobar la validez de cada solución generada y, en caso de no ser válida, repararla o descartarla.

Decidimos cambiar a la representación por permutación porque garantiza la restricción por construcción, ya que es siempre una reordenación de las mismas 30 tomas, cortada en bloques fijos de 6. Esto imposibilita generar una solución inválida y supone que cualquier intercambio de dos posiciones produce siempre otra solución válida. Por lo tanto, encaja directamente con nuestra solución pretendida de búsqueda local, donde vamos a generar y evaluar muchas soluciones vecinas, ahorrándonos comprobar la factibilidad en cada paso.

Sin embargo, el coste de este cambio es que la representación se vuelve redundante debido a que el orden de las tomas dentro de un mismo día no afecta al coste. Como resultado, varias permutaciones distintas codifican la misma solución real. Esto es algo a tener en cuenta al definir cómo se explora el espacio de soluciones, como se detallará más adelante.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

## Función Objetivo

Antes de nada, identificamos los parámetros que deben aparecer en la función objetivo o que tengan influencia en la misma:

* Conjunto de tomas: $T = \{1, 2, \dots, 30\}$ Este conjunto tiene todas las tomas a grabar, empezando en la toma 1 y acabando en la 30.
* Conjunto de actores: $A = \{1, 2,3,4,5,6,7,8,9, 10\}$ Conjunto de actores, 10 en concreto según el enunciado.
* Conjunto de días de rodaje: En el peor caso podría haber hasta 30 días (una toma por día), pero fijamos $D = \{1,2,3,4,5\}$ como simplificación de modelado para acotar el espacio de búsqueda, ya que es el mínimo número de días que permite respetar la restricción de máximo 6 tomas por día. No se descarta que más días dieran menor coste, pero lo asumimos por tratabilidad.
* Matriz de datos de entrada: $M$.
* Matriz de datos de salida: $S$. Se corresponde con un reparto de las 30 tomas entre los 5 días (en nuestra representación, la permutación troceada en bloques de 6).

El objetivo del problema es minimizar el coste total de desplazamientos de los actores. Para ello, definimos la función objetivo con su variable de asistencia al set de grabación ($Y_{d,a} = 1$ si existe alguna toma $t$ asignada al día $d$ en $S$ tal que $M[t,a] = 1$, si no, $0$).



$$\min \quad f(S) = \sum_{d=1}^{5} \sum_{a=1}^{10} Y_{d,a}$$

Esta función calcula el coste total de desplazamientos para una solución S dada, y buscamos la S que lo minimiza.

## ¿Minimización o Maximización?

Como se ha comentado previamente, este problema se corresponde con un problema de minimización, puesto que el objetivo es reducir el coste de la producción a la hora de coordinar el doblaje, no aumentarlo. Maximizar el coste sería trivial (bastaría con esparcir al máximo a los actores en días distintos) y no tiene sentido desde el punto de vista económico.


In [27]:
def funcion_objetivo(S, datos=tomas, D=5, tomas_dia=MAX_TOMAS_DIA):
  """
  La funcion evalua una solucion candidata, representada como una permutacion de las 30 tomas (array de indices 0..29) y
  troceada en bloques de 6 dias. Calcula f(S) como suma de Y_{d,a} para todo dia d y actor a, es decir, el numero total
  de desplazamientos por los que hay que pagar.
  """
  coste = 0
  for d in range(D):
      bloque = S[d*tomas_dia : (d+1)*tomas_dia]
      Y_d = datos[bloque].any(axis=0)
      coste += Y_d.sum()
  return coste

# Se presenta un ejemplo simple con la permutacion identidad (6 primeras tomas el dia 1, 6 siguientes el dia 2, ...)
S_ejemplo = np.arange(NUM_TOMAS)
print(f"f(S) con la permutacion identidad: {funcion_objetivo(S_ejemplo)}")

# Finalmente comprobamos que la S especificada afecta al resultado, con varias permutaciones aleatorias
for semilla in range(3):
    aleatorios = np.random.default_rng(semilla)
    S_aleatoria = aleatorios.permutation(NUM_TOMAS)
    print(f"f(S) con permutacion aleatoria {semilla}: {funcion_objetivo(S_aleatoria)}")

f(S) con la permutacion identidad: 38
f(S) con permutacion aleatoria 0: 38
f(S) con permutacion aleatoria 1: 40
f(S) con permutacion aleatoria 2: 37


Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

Dado el modelo elegido, el algoritmo de fuerza bruta consiste en generar sistemáticamente todos los repartos posibles y evaluar la función objetivo en cada uno, quedándose con el que obtenga un menor coste.

Para generarlos sin repetir evaluaciones en reordenaciones internas de cada día que no cambian el coste (la redundancia de (6!)^5 comentada en preguntas anteriores), el algoritmo procede día a día. El primer día elige qué 6 tomas de las 30 se le asignan (C(30,6) posibilidades), el segundo día elige 6 de las 24 tomas restantes (C(24,6) posibilidades), y así sucesivamente hasta agotar las tomas. El producto de estas elecciones genera exactamente las 30!/(6!)^5 agrupaciones distintas que se calcularon previamente, sin repetir ninguna.

Al ser exhaustivo, el algoritmo garantiza encontrar el óptimo global, en contraposición a los algoritmos heurísticos por ejemplo. Sin embargo, el precio de esa garantía es el coste computacional, que se analiza en la siguiente pregunta. Como evaluar las 30!/(6!)^5 agrupaciones del problema completo es inviable en un tiempo razonable, validamos el algoritmo sobre una versión reducida (12 tomas repartidas en 2 días de 6), dejando la prueba completa para el análisis de complejidad.

In [28]:
def generar_particiones(indices, D, tomas_dia):
    """Funcion generador que obtiene todas las formas posibles de repartir indices en D grupos etiquetados de tamaño tomas_dia"""
    if D == 1:
        yield [tuple(indices)]
        return
    # se recorren las combinaciones posibles y las restantes se guardan con una list comprehension
    for combo in itertools.combinations(indices, tomas_dia):
        resto = [i for i in indices if i not in combo]
        for resto_part in generar_particiones(resto, D - 1, tomas_dia):
            yield [combo] + resto_part

def fuerza_bruta(datos, D, tomas_dia):
    "Aplicacion del algoritmo de fuerza bruta quedandonos con la mejor solucion y el mejor coste"
    n = datos.shape[0]
    mejor_coste, mejor_solucion = None, None
    for grupos in generar_particiones(list(range(n)), D, tomas_dia):
        # se concatenan los grupos y se reutiliza la funcion_objetivo descrita en el apartado anterior
        S = np.concatenate(grupos)
        coste = funcion_objetivo(S, datos=datos, D=D, tomas_dia=tomas_dia)
        if mejor_coste is None or coste < mejor_coste:
            mejor_coste, mejor_solucion = coste, grupos
    return mejor_solucion, mejor_coste

# Prueba sobre un conjunto reducido: 12 primeras tomas y 2 días de 6
datos_min = tomas[:12]
solucion, coste = fuerza_bruta(datos_min, D=2, tomas_dia=6)
print(f"Combinaciones evaluadas: {comb(12,6)}")
print(f"Mejor solucion: {solucion}")
print(f"Coste optimo: {coste}")

Combinaciones evaluadas: 924
Mejor solucion: [(0, 1, 2, 3, 4, 10), (5, 6, 7, 8, 9, 11)]
Coste optimo: 14


Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

El número de posibles soluciones (30!/(6!)^5, calculado anteriormente) es una propiedad del problema, no de este algoritmo. La complejidad, en cambio, es el número de operaciones elementales que el método de fuerza bruta realiza para recorrerlas todas: el resultado de multiplicar ese número de soluciones por el coste de evaluar cada una.

Para determinar el coste de evaluar cada solución debemos considerar que la función objetivo recorre los D días, y para cada día comprueba qué actores participan mirando las k tomas de ese día en las A=10 columnas de actores. Eso es O(k·A) por día, y como hay D días con D·k=N, el coste total de evaluar una solución es O(N·A).

Complejidad total, multiplicando ambos factores:

> T(N) = O( N!/(k!)^(N/k) · N · A )

Con k=6 y A=10 fijos como constantes del problema, esta expresión crece de forma muy superior a cualquier exponencial de base fija. En otras palabras, la complejidad queda acotada superiormente por O(N!) pero es bastante más ajustada que N! sin más, porque el (k!)^(N/k) del denominador descuenta las reordenaciones internas de cada día. En cualquier caso, es una complejidad no polinómica, del orden factorial: crece demasiado rápido para ser tratable en la práctica según aumenta N.

Además de la cota teórica mencionada, pasamos a medir el algoritmo en una instancia reducida del mismo problema (12 tomas). Como cada evaluación realiza el mismo tipo de operación, asumimos que el tiempo por evaluación es aproximadamente constante, de forma que el número de evaluaciones domina el tiempo de ejecución, tal como indica la fórmula. Extrapolando esa tasa medida al problema completo (30 tomas), la ejecución tardaría varios millones de años, lo cual evidencia la necesidad de implementar un algoritmo distinto para resolver el problema en tiempo razonable.

In [36]:

# Medicion sobre instancia reducida
t0 = time.time()
_, _ = fuerza_bruta(datos_min, D=2, tomas_dia=6)
t1 = time.time()
# Evaluaciones realizadas y tiempo necesario en la instancia reducida
evals_12 = comb(12, 6)
tiempo_12 = t1 - t0
print(f"N=12: {evals_12} evaluaciones, tiempo={tiempo_12:.4f}s")

# Extrapolacion teorica a 30 tomas haciendo uso del dato medido
A = NUM_ACTORES
c = tiempo_12 / (evals_12 * 12 * A)  # constante de tiempo por operacion elemental
evals_30 = factorial(30) // (factorial(6) ** 5)
T30 = evals_30 * 30 * A * c

print(f"\nEvaluaciones necesarias para 30 tomas: {evals_30:,}")
print(f"Tiempo estimado para 30 tomas por fuerza bruta: {T30:.2e} segundos (~{T30/3600/24/365:,.0f} años)")

N=12: 924 evaluaciones, tiempo=0.0226s

Evaluaciones necesarias para 30 tomas: 1,370,874,167,589,326,400
Tiempo estimado para 30 tomas por fuerza bruta: 8.39e+13 segundos (~2,660,004 años)


(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

Para mejorar la complejidad del algoritmo por fuerza bruta diseñamos un algoritmo de búsqueda local, en el que partimos de una solución cualquiera (una permutación de las 30 tomas en bloques de 6) y la vamos mejorando por pasos pequeños, en lugar de examinar todas las soluciones posibles.

El vecindario de una solución es el conjunto de soluciones que se obtienen intercambiando de posición dos tomas que estén en días distintos. No se consideran los intercambios entre tomas del mismo día porque, como ya vimos, no cambian el coste (el orden dentro de un día es irrelevante), así que incluirlos sería repetir evaluaciones. En cada iteración, el algoritmo evalúa todo el vecindario y se mueve al vecino de menor coste si mejora la solución actual, y si ningún vecino mejora, el algoritmo se detiene puesto que significa que ha alcanzado un óptimo local.

Como la búsqueda local puede quedar atrapada en un óptimo local que no sea el óptimo global, se ejecuta varias veces desde soluciones iniciales aleatorias distintas (reinicios múltiples) y se conserva la mejor de todas las ejecuciones.

La razón por la que mejora al algoritmo por fuerza bruta es que mientras que el primero evalúa todas las agrupaciones posibles, la búsqueda local evalúa un espacio mucho menor (vecindario de la solución actual). En números, la diferencia es de 30!/(6!)^5 frente a un tamaño de orden N² (~1,37·10^18 vs 360). El número de iteraciones hasta alcanzar un óptimo local es, en la práctica, muy pequeño: en nuestras pruebas, entre 3 y 8 iteraciones por ejecución. Esto convierte un problema de orden factorial en uno que, para un número fijo de reinicios, es polinómico en N.

Sin embargo, el precio a pagar por esta mejora es que se pierde la garantía de optimalidad. Esto quiere decir que mientras que fuerza bruta siempre encuentra el óptimo global, la búsqueda local solo garantiza un óptimo local, que puede o no coincidir con el global. Esto se compensa parcialmente con los reinicios múltiples, aunque sin garantía formal.

Validamos que el algoritmo funciona correctamente comparándolo con el óptimo exacto que ya conocíamos por fuerza bruta en la instancia reducida de 12 tomas (coste óptimo 14): la búsqueda local lo alcanza en el 100% de las ejecuciones probadas.

In [42]:
def busqueda_local(datos, D, tomas_dia, rng, max_iter=200):
    """Aplicacion de busqueda local -> se parte de una permutacion aleatoria y se mejora intercambiando
    tomas de dias distintos, hasta alcanzar un optimo local"""
    N = D * tomas_dia
    # solucion inicial: una permutacion al azar de las N tomas
    permutacion = rng.permutation(N)
    mejor_coste = funcion_objetivo(permutacion, datos, D, tomas_dia)
    # se itera recorriendo los vecinos de la solucion actual, buscando una mejor,
    # hasta que ningun vecino mejore (optimo local) o se alcance el limite de iteraciones
    mejora = True
    num_iteraciones = 0
    while mejora and num_iteraciones < max_iter:
        mejora = False
        # se guardan aqui el mejor intercambio de esta vuelta y el coste que produce
        mejor_intercambio, mejor_c = None, mejor_coste

        # se recorren todos los pares de posiciones posibles (i, j)
        for i in range(N):
            for j in range(i + 1, N):
                if i // tomas_dia == j // tomas_dia:
                    continue  # mismo dia -> intercambio neutro, no cambia el coste, se descarta

                # se prueba el intercambio, se evalua, y se deshace para seguir probando otros
                permutacion[i], permutacion[j] = permutacion[j], permutacion[i]
                c = funcion_objetivo(permutacion, datos, D, tomas_dia)
                if c < mejor_c:
                    mejor_c, mejor_intercambio = c, (i, j)
                permutacion[i], permutacion[j] = permutacion[j], permutacion[i]  # se deshace

        # si se encuentra algun intercambio que mejora, se aplica de forma definitiva
        if mejor_intercambio is not None:
            i, j = mejor_intercambio
            permutacion[i], permutacion[j] = permutacion[j], permutacion[i]
            mejor_coste = mejor_c
            mejora = True  # se sigue intentando mejorar en la siguiente vuelta

        num_iteraciones += 1

    return permutacion, mejor_coste


# validacion para 12 tomas (optimo conocido es 14)
resultados_12 = []
for s in range(10):
    aleatorios = np.random.default_rng(s)
    _, c = busqueda_local(tomas[:12], D=2, tomas_dia=6, rng=aleatorios)
    resultados_12.append(int(c))
print("Optimo real obtenido por fuerza bruta: 14")
print("Resultados busqueda local (10 semillas):", resultados_12)

# problema completo -> 30 tomas con reinicios multiples
mejores = []
for s in range(30):
    aleatorios = np.random.default_rng(s)
    _, c = busqueda_local(tomas, D=5, tomas_dia=6, rng=aleatorios)
    mejores.append(int(c))

print(f"\nMejor coste encontrado con 30 reinicios: {min(mejores)}")
print(f"Costes obtenidos: {sorted(mejores)}")

Optimo real obtenido por fuerza bruta: 14
Resultados busqueda local (10 semillas): [14, 14, 14, 14, 14, 14, 14, 14, 14, 14]

Mejor coste encontrado con 30 reinicios: 29
Costes obtenidos: [29, 29, 29, 29, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 31, 31, 31, 31, 31, 31, 31, 31, 32, 32, 32, 33, 33]


(*)Calcula la complejidad del algoritmo

Respuesta

Al igual que en fuerza bruta, la complejidad de la búsqueda local es el número de operaciones elementales que realiza, no el número de soluciones que existen en el espacio del problema. Aquí hay tres factores que multiplicar, no dos: el tamaño del vecindario que se examina en cada iteración, el coste de evaluar cada vecino, y el número de iteraciones hasta converger.

En cuanto al tamaño del vecindario, en cada iteración se recorren todos los pares de posiciones (i, j)  y se descartan los del mismo día. El resultado es un vecindario de tamaño O(N²).

Cada evaluación llama a funcion_objetivo, que recorre los D días comprobando la participación de los A=10 actores: O(N·A) por evaluación, igual que en fuerza bruta.

En el número de iteraciones, hay una cota demostrable: la función objetivo es un entero acotado entre un mínimo y un máximo de D·A (todos los actores todos los días), y cada iteración de la búsqueda local reduce el coste en al menos 1 o se detiene. Por lo tanto, el número de iteraciones nunca puede superar D·A.

La complejidad total de una ejecución se obtiene multiplicando los tres factores:

> T(N) = O(D·A · N² · N·A) = O(D · A² · N³) = O(N⁴)

Finalmente se obtiene O(N⁴), ya que D = N/k con k constante. Esto es una complejidad polinómica, muy inferior a la complejidad factorial de fuerza bruta (N!/(k!)^(N/k)). Al ejecutar varios reinicios (R, una constante fija que elegimos nosotros, no crece con N), la complejidad total queda en O(R·N⁴), que sigue siendo polinómica.

Medimos el algoritmo con N=12, 18, 24 y 30. El tiempo de ejecución crece de forma claramente superior a la lineal según aumenta N (se mantiene siempre en el orden de milisegundos, pero aumenta varias decenas de veces entre el tamaño más pequeño y el más grande), lo cual es compatible con el crecimiento de orden N⁴ obtenido en la fórmula teórica. El número de iteraciones real se mantuvo en todos los casos muy por debajo de la cota teórica D·A.

In [56]:
A = NUM_ACTORES

# se prueban distintos tamaños de N, manteniendo fijo el tope de 6 tomas/dia,
# para comprobar como escala el tiempo de ejecucion con N
duraciones_por_tam = []
for num_tomas_test in [12, 18, 24, 30]:
    dias_test = num_tomas_test // 6
    datos_test = tomas[:num_tomas_test]

    # se repite varias veces con distintas semillas, para no depender de una sola ejecucion
    duraciones = []
    for s in range(5):
        rng = np.random.default_rng(s)
        t0 = time.time()
        permutacion, coste_final = busqueda_local(datos_test, dias_test, 6, rng)
        t1 = time.time()
        duraciones.append(t1 - t0)

    cota_teorica = dias_test * A  # cota maxima de iteraciones demostrada en el texto (D*A)
    duraciones_por_tam.append((num_tomas_test, np.mean(duraciones)))
    print(f"N={num_tomas_test}: tiempo medio={np.mean(duraciones):.4f}s, cota D*A={cota_teorica}")

N=12: tiempo medio=0.0026s, cota D*A=20
N=18: tiempo medio=0.0105s, cota D*A=30
N=24: tiempo medio=0.0321s, cota D*A=40
N=30: tiempo medio=0.0682s, cota D*A=50


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Para que el juego de datos tenga sentido, no nos hemos limitado a generar unos y ceros con probabilidad fija. Dado que en los datos reales, la participación de los actores está muy repartida de forma desigual, hemos definido un generador que asigna a cada actor una probabilidad de participación distinta y decide la participación de cada toma como un sorteo con esa probabilidad. Después corrige dos casos sin sentido en el problema real: una toma sin actores, o un actor sin ninguna toma. El número de tomas debe ser múltiplo de 6, para mantener el modelo de días completos usado en todo el proyecto.

Respuesta

In [57]:
def generar_datos_aleatorios(n_tomas, n_actores, semilla=None):
    """
    Genera una matriz [tomas x actores] realista: pocos actores protagonistas y
    varios secundarios que aparecen en pocas, imitando la distribucion de los
    datos reales proporcionados.
    """
    # el numero de tomas debe ser multiplo de 6, para mantener el modelo de dias completos
    assert n_tomas % MAX_TOMAS_DIA == 0, "n_tomas debe ser multiplo de MAX_TOMAS_DIA"
    aleatorios = np.random.default_rng(semilla)

    # se asigna a cada actor una probabilidad de participacion distinta
    prob_actor = aleatorios.beta(a=1.2, b=3.0, size=n_actores) * 0.75 + 0.05

    # se decide la participacion de cada toma como un sorteo con la probabilidad de cada actor
    datos = aleatorios.random((n_tomas, n_actores)) < prob_actor

    # se corrigen los dos casos sin sentido en el problema real
    for fila in np.where(~datos.any(axis=1))[0]:       # ninguna toma sin actor
        datos[fila, aleatorios.integers(0, n_actores)] = True
    for columna in np.where(~datos.any(axis=0))[0]:     # ningun actor sin toma
        datos[aleatorios.integers(0, n_tomas), columna] = True

    return datos

# instancia del mismo tamaño que el problema real, para poder comparar
datos_generados = generar_datos_aleatorios(NUM_TOMAS, NUM_ACTORES, semilla=42)
print(f"Participacion por actor: {datos_generados.mean(axis=0).round(2)}")
print(f"Actores por toma (media): {datos_generados.sum(axis=1).mean():.2f}")

Participacion por actor: [0.3  0.2  0.37 0.2  0.57 0.5  0.33 0.07 0.37 0.43]
Actores por toma (media): 3.33


Aplica el algoritmo al juego de datos generado

Respuesta

Aplicamos la búsqueda local sobre el juego de datos generado, del mismo tamaño que el problema real. Para contextualizar el resultado, lo comparamos con un baseline de 500 agrupaciones aleatorias sin optimizar sobre esos mismos datos.

El baseline aleatorio da un coste entre 39 y 46. La búsqueda local encuentra un coste de 32 en el mejor de los 30 reinicios, una mejora clara y del mismo orden que la obtenida sobre los datos reales, lo que confirma que el algoritmo generaliza bien a datos distintos de los originales, no solo a la instancia concreta con la que se diseñó.

In [59]:
datos_generados = generar_datos_aleatorios(NUM_TOMAS, NUM_ACTORES, semilla=42)

# baseline sin optimizar: coste de agrupaciones aleatorias, para poder comparar
aleatorios = np.random.default_rng(0)
costes_aleatorios = [
    funcion_objetivo(aleatorios.permutation(NUM_TOMAS), datos_generados, D=5, tomas_dia=6)
    for _ in range(500)
]
print(f"Baseline aleatorio: min={min(costes_aleatorios)}, max={max(costes_aleatorios)}, media={np.mean(costes_aleatorios):.1f}")

# busqueda local con reinicios multiples, sobre los datos generados
mejores_generados = []
for s in range(30):
    aleatorios = np.random.default_rng(s)
    _, c = busqueda_local(datos_generados, D=5, tomas_dia=6, rng=rng)
    mejores_generados.append(int(c))

print(f"Busqueda local (30 reinicios): mejor coste={min(mejores_generados)}")


Baseline aleatorio: min=39, max=46, media=43.3
Busqueda local (30 reinicios): mejor coste=32


Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

- Material de la asignatura (presentaciones 03MIAR - Algoritmos de Optimización, VIU).
- León Martínez, Á. Algoritmo de búsqueda local iterada para la secuenciación en talleres de flujo y minimización de makespan. Proyecto Fin de Grado, Universidad de Sevilla. https://ereding.etsi.us.es/bibing/proyectos/abreproy/91710/fichero/TFG-1710-LEON.pdf
- Martí Cunquero, R. Algoritmos Heurísticos en Optimización Combinatoria. Universitat de València. http://yalma.fime.uanl.mx/~roger/work/teaching/mecbs5122/1-Introduction/Intro-by-Rafa%20Marti.pdf
- Se ha utilizado un asistente de IA como apoyo en la búsqueda y formato de las referencias académicas consultadas, la construcción en LaTeX de las expresiones matemáticas y la depuración de errores en el código.

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

Identificamos varias líneas para ampliar este trabajo. La más directa sería tratar el número de días como una variable más, en vez de fijarlo en 5, ya que permitiría agrupaciones con algún día no lleno que quizá redujeran el coste, aunque complicaría tanto el conteo combinatorio como la representación de la solución.

También cabría probar metaheurísticas más sofisticadas que el simple reinicio aleatorio, como el recocido simulado o la búsqueda tabú, que aceptan peores soluciones temporales de forma controlada y suelen escapar mejor de óptimos locales, sin necesidad de cambiar la representación ni la función objetivo que ya tenemos.

El modelo también podría enriquecerse con costes de desplazamiento distintos por actor, disponibilidad restringida en ciertos días, prioridades entre tomas, o incluso dividir una toma entre varias sesiones. Y en cuanto al tamaño, ya hemos visto que fuerza bruta es inviable más allá de instancias muy pequeñas mientras que la búsqueda local escala razonablemente bien. Para tamaños mucho mayores, ayudaría tanto evaluar cada intercambio de forma incremental (recalculando solo los dos días afectados, en vez de la solución completa) como ejecutar los distintos reinicios en paralelo, ya que son independientes entre sí.